## Milestone 3 — Evaluation Strategy for Fraud Detection

In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.isotonic import IsotonicRegression

# Seed for reproducibility
np.random.seed(42)

# Simulating 100,000 card transactions with 0.1% fraud (only 100 frauds)
N = 100000
n_fraud = int(N * 0.001)
n_legit = N - n_fraud

y_true = np.zeros(N, dtype=int)
y_true[:n_fraud] = 1

# Generate synthetic risk scores: legit scores are low, fraud scores are higher
scores_legit = np.random.beta(0.5, 20.0, size=n_legit)
scores_fraud = np.random.beta(2.0, 5.0, size=n_fraud)
y_scores = np.concatenate([scores_fraud, scores_legit])


**1. Metric Comparison: Why Accuracy & ROC-AUC are Misleading**

In [4]:
# Accuracy trap: A dumb model predicting all 0s gets 99.9% accuracy while catching 0 frauds!
# ROC-AUC trap: Because legitimate pool is huge (99,900), 4,000 false alarms still gives FPR < 5%,
# making ROC look amazing (0.97+) even though operational precision is terrible (~2%).

roc = roc_auc_score(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)

print(f"Total Transactions: {N:,} | Fraud count: {n_fraud} (0.1%)")
print(f"ROC-AUC: {roc:.4f}  <-- Deceptively high because of the 99,900 true negatives!")
print(f"PR-AUC:  {pr_auc:.4f}  <-- Real measure of performance on the rare fraud class")

# At a standard threshold of 0.10:
preds = (y_scores >= 0.10).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"At threshold 0.10: Caught {tp}/{n_fraud} frauds, but triggered {fp:,} false alarms!")
print(f"Operational Precision: {precision*100:.2f}% (over 97% of flagged alerts are innocent users)")


Total Transactions: 100,000 | Fraud count: 100 (0.1%)
ROC-AUC: 0.9789  <-- Deceptively high because of the 99,900 true negatives!
PR-AUC:  0.5318  <-- Real measure of performance on the rare fraud class
At threshold 0.10: Caught 86/100 frauds, but triggered 4,101 false alarms!
Operational Precision: 2.05% (over 97% of flagged alerts are innocent users)


**2. Cost-Optimal Decision Threshold & Calibration**

In [6]:
# Business costs:
# False Negative (missed fraud) = $200
# False Positive (customer friction / 2FA SMS / review) = $5
# Math: Flag if cost(flag) < cost(pass) --> 5 * (1-p) < 200 * p  -->  p > 5 / 205 = 2.44%

C_fn = 200.0
C_fp = 5.0
optimal_th = C_fp / (C_fn + C_fp)

# Let's compare total costs across different thresholds on a validation set
th_range = [0.01, 0.0244, 0.05, 0.10, 0.20, 0.50]
print(f"Theoretical Cost-Optimal Threshold: {optimal_th:.4f} ({optimal_th*100:.2f}%)\n")
print(f"{'Threshold':<12} | {'Missed (FN)':<12} | {'False Alarms (FP)':<18} | {'Total Dollar Cost'}")
print("-" * 65)

for th in th_range:
    p_binary = (y_scores >= th).astype(int)
    fn = np.sum((p_binary == 0) & (y_true == 1))
    fp = np.sum((p_binary == 1) & (y_true == 0))
    loss = (fn * C_fn) + (fp * C_fp)
    print(f"{th:<12.4f} | {fn:<12} | {fp:<18} | ${loss:,.2f}")

# Score Calibration check (Brier score)
brier_raw = np.mean((y_scores - y_true) ** 2)
# Recalibrating using Isotonic Regression
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(y_scores, y_true)
calibrated_scores = iso.predict(y_scores)
brier_cal = np.mean((calibrated_scores - y_true) ** 2)

print("-" * 65)
print(f"Raw Brier Score:        {brier_raw:.6f}")
print(f"Calibrated Brier Score: {brier_cal:.6f} (Improved by {(brier_raw - brier_cal)/brier_raw*100:.2f}%)")


Theoretical Cost-Optimal Threshold: 0.0244 (2.44%)

Threshold    | Missed (FN)  | False Alarms (FP)  | Total Dollar Cost
-----------------------------------------------------------------
0.0100       | 0            | 52911              | $264,555.00
0.0244       | 0            | 32374              | $161,870.00
0.0500       | 4            | 15390              | $77,750.00
0.1000       | 14           | 4101               | $23,305.00
0.2000       | 29           | 320                | $7,400.00
0.5000       | 90           | 0                  | $18,000.00
-----------------------------------------------------------------
Raw Brier Score:        0.002235
Calibrated Brier Score: 0.000603 (Improved by 73.04%)


**3. Detecting Production Drift with Population Stability Index (PSI)**

In [8]:
# Labels take 30-60 days to arrive, so we can't wait for accuracy to drop to detect drift!
# Unsupervised telemetry: PSI checks if incoming feature distributions shifted vs baseline.

def get_psi(expected, actual, bins=10):
    edges = np.percentile(expected, np.linspace(0, 100, bins + 1))
    edges[0] -= 1e-5; edges[-1] += 1e-5
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = (e_counts + 1e-4) / np.sum(e_counts + 1e-4)
    a_pct = (a_counts + 1e-4) / np.sum(a_counts + 1e-4)
    return np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))

# Baseline training feature (e.g. transaction amounts)
base_amounts = np.random.normal(loc=100.0, scale=25.0, size=10000)
# Normal production traffic (Week 1)
prod_normal = np.random.normal(loc=100.5, scale=24.8, size=5000)
# Shifted production traffic (Week 4: new fraud ring charging unusual amounts)
prod_drifted = np.random.normal(loc=135.0, scale=40.0, size=5000)

psi_clean = get_psi(base_amounts, prod_normal)
psi_alert = get_psi(base_amounts, prod_drifted)

print(f"Normal traffic PSI:  {psi_clean:.4f}  --> Status: Healthy (< 0.10)")
print(f"Shifted traffic PSI: {psi_alert:.4f}  --> Status: DRIFT DETECTED (>= 0.25) -> Trigger Retraining!")


Normal traffic PSI:  0.0031  --> Status: Healthy (< 0.10)
Shifted traffic PSI: 0.9277  --> Status: DRIFT DETECTED (>= 0.25) -> Trigger Retraining!
